# Wildfire Training Notebook (v2)

Train next-day California wildfire risk on Kaggle Input packs (`knn` or `median`).

Flow: configure → validate → preprocess → features → split → pipeline + Optuna → train → validate/calibrate → feature importance → export.

**2025 / `test.parquet` is not scored here** — use the inference notebook.


## 1. Load, Import & Configuration

Set `DATA_SOURCE` (`knn` / `median`) and `CELL_SUBSET`. Uses fixed Kaggle paths; prefers `train.parquet` + `val.parquet`.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import math
import os
import platform
import subprocess
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "wildfire-mpl"))
os.environ.setdefault("MPLBACKEND", "Agg")

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import sklearn
from sklearn import set_config
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer

SEED = 42

def env_get(*names: str, default: str = "") -> str:
    for name in names:
        value = os.environ.get(name)
        if value is not None and str(value).strip() != "":
            return str(value).strip()
    return default

# knn → california-wildfire-knn | median → california-wildfire-median
DATA_SOURCE = "knn"  # change to "median" for the median pack
DATA_SOURCE = env_get("WILDFIRE_DATA_SOURCE", "CHAMPION_DATA_SOURCE", default=DATA_SOURCE).lower()
if DATA_SOURCE in {"stage_c_knn", "knn"}:
    DATA_SOURCE = "knn"
elif DATA_SOURCE in {"stage_c", "median"}:
    DATA_SOURCE = "median"
else:
    raise ValueError("DATA_SOURCE must be 'knn' or 'median'")

_DATA_PATHS = {
    "knn": "/kaggle/input/datasets/lakshay654/california-wildfire-knn",
    "median": "/kaggle/input/datasets/lakshay654/california-wildfire-median",
}
DATA_DIRECTORY = env_get("WILDFIRE_DATA_DIR", "CHAMPION_DATA_DIR", default=_DATA_PATHS[DATA_SOURCE])
TRAINING_DATA_STAGE = "stage_c_knn" if DATA_SOURCE == "knn" else "stage_c"
USE_PRECOMPUTED_KNN = DATA_SOURCE == "knn"
IMPUTATION_METHOD = "precomputed KNN" if USE_PRECOMPUTED_KNN else "training-only median"

CELL_SUBSET = "all"  # all | high_fire | high_medium_fire
CELL_SUBSET = env_get("WILDFIRE_CELL_SUBSET", "CHAMPION_CELL_SUBSET", default=CELL_SUBSET)
if CELL_SUBSET not in {"all", "high_fire", "high_medium_fire"}:
    raise ValueError("CELL_SUBSET must be 'all', 'high_fire', or 'high_medium_fire'")
CELL_SUBSET_CATEGORIES = {
    "all": None,
    "high_fire": ["High Outlier", "High"],
    "high_medium_fire": ["High Outlier", "High", "Medium"],
}
FIRE_REGION_CSV = env_get(
    "WILDFIRE_FIRE_REGION_CSV",
    "CHAMPION_FIRE_REGION_CSV",
    default="/kaggle/input/datasets/lakshay654/firms-test/fire_analysis2.csv",
)
USE_SPLIT_FILES = env_get("WILDFIRE_USE_SPLIT_FILES", default="1").lower() not in {"0", "false", "no"}

MODE = env_get("WILDFIRE_MODE", "CHAMPION_MODE", default="full").lower()
if MODE not in {"full", "smoke"}:
    raise ValueError("MODE must be 'full' or 'smoke'")
USE_GPU = env_get("WILDFIRE_USE_GPU", "CHAMPION_USE_GPU", default="1").lower() not in {"0", "false", "no"}
np.random.seed(SEED)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
set_config(display="diagram")

def show(value: Any) -> None:
    try:
        from IPython.display import display
        display(value)
    except ImportError:
        print(value)

def show_image(path: Path) -> None:
    try:
        from IPython.display import Image, display
        display(Image(filename=str(path)))
    except ImportError:
        print(f"Saved image: {path}")

def pack_paths(root: Path) -> dict[str, Path] | None:
    root = root.resolve()
    flat_meta = root / "dataset_metadata.json"
    nested_meta = root / "metadata" / "dataset_metadata.json"
    if flat_meta.is_file():
        meta = flat_meta
        features = root / "feature_columns.json"
    elif nested_meta.is_file():
        meta = nested_meta
        features = root / "metadata" / "feature_columns.json"
    else:
        return None
    has_table = (root / "all.parquet").is_file() or (
        (root / "train.parquet").is_file() and (root / "val.parquet").is_file()
    )
    if not has_table or not (root / "meta.json").is_file() or not features.is_file():
        return None
    try:
        stage = json.loads(meta.read_text(encoding="utf-8")).get("stage")
    except (OSError, json.JSONDecodeError):
        return None
    if stage != TRAINING_DATA_STAGE:
        return None
    return {
        "root": root,
        "meta": root / "meta.json",
        "dataset_metadata": meta,
        "features": features,
        "all": root / "all.parquet",
        "train": root / "train.parquet",
        "val": root / "val.parquet",
        "test": root / "test.parquet",
    }

def resolve_data_pack() -> dict[str, Path]:
    candidates = [
        Path(DATA_DIRECTORY).expanduser(),
        Path(_DATA_PATHS[DATA_SOURCE]),
        Path.cwd(),
        Path.cwd() / ("stage_c_knn" if DATA_SOURCE == "knn" else "stage_c"),
    ]
    # Local Milestone 5 flat packs and Milestone 4 shared cache.
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        candidates.append(base / "Milestone 5" / "kaggle_datasets" / ("stage_c_knn" if DATA_SOURCE == "knn" else "stage_c"))
        candidates.append(base / "Milestone 4" / "numerical_nextday" / "outputs" / "m4_shared_cache" / ("stage_c_knn" if DATA_SOURCE == "knn" else "stage_c"))
    checked = []
    for root in candidates:
        checked.append(str(root))
        found = pack_paths(root)
        if found is not None:
            return found
    raise FileNotFoundError(
        f"Could not find {DATA_SOURCE} pack (stage={TRAINING_DATA_STAGE}). "
        f"Set DATA_DIRECTORY. Tried:\n" + "\n".join(f"  - {item}" for item in checked)
    )

DATA_PACK = resolve_data_pack()
DATA_ROOT = DATA_PACK["root"]
META_JSON = DATA_PACK["meta"]
DATASET_METADATA_JSON = DATA_PACK["dataset_metadata"]
FEATURE_COLUMNS_JSON = DATA_PACK["features"]
ALL_PARQUET = DATA_PACK["all"]
TRAIN_PARQUET = DATA_PACK["train"]
VAL_PARQUET = DATA_PACK["val"]
TEST_PARQUET = DATA_PACK["test"]

USE_SPLIT = bool(
    USE_SPLIT_FILES and TRAIN_PARQUET.is_file() and VAL_PARQUET.is_file()
)
if USE_SPLIT:
    LOAD_MODE = "train+val split files"
    TABLE_PARQUETS = [TRAIN_PARQUET, VAL_PARQUET]
else:
    if not ALL_PARQUET.is_file():
        raise FileNotFoundError(f"Need train+val parquet or all.parquet under {DATA_ROOT}")
    LOAD_MODE = "all.parquet (years <= 2024)"
    TABLE_PARQUETS = [ALL_PARQUET]

default_output = (
    Path(f"/kaggle/working/wildfire_training_outputs_{DATA_SOURCE}_{CELL_SUBSET}")
    if Path("/kaggle/working").is_dir()
    else Path.cwd() / "notebook_outputs" / f"wildfire_training_{DATA_SOURCE}_{CELL_SUBSET}_{MODE}"
)
OUTPUT_DIR = Path(env_get("WILDFIRE_OUTPUT_DIR", "CHAMPION_OUTPUT_DIR", default=str(default_output))).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "metrics").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)

def reported_gpu() -> str | None:
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            check=True, capture_output=True, text=True, timeout=10,
        )
        values = [line.strip() for line in result.stdout.splitlines() if line.strip()]
        return ", ".join(values) if values else None
    except (FileNotFoundError, subprocess.SubprocessError):
        return None

def lightgbm_device() -> tuple[str, dict[str, str]]:
    if not USE_GPU:
        return "cpu", {"cpu": "GPU disabled"}
    rng = np.random.default_rng(SEED)
    x = rng.normal(size=(2000, 20)).astype("float32")
    y = rng.integers(0, 2, size=2000, dtype="int8")
    failures: dict[str, str] = {}
    for device in ("cuda", "gpu"):
        try:
            dataset = lgb.Dataset(x, label=y)
            lgb.train(
                {"objective": "binary", "device_type": device, "verbosity": -1, "seed": SEED},
                dataset, num_boost_round=2,
            )
            return device, failures
        except Exception as error:
            failures[device] = str(error).splitlines()[-1][:400]
    return "cpu", failures

DEVICE, DEVICE_FAILURES = lightgbm_device()
GPU_NAME = reported_gpu()

print("=" * 72)
print("WILDFIRE TRAINING CONFIGURATION")
print("=" * 72)
print(f"Data source:       {DATA_SOURCE} ({TRAINING_DATA_STAGE})")
print(f"Cell subset:       {CELL_SUBSET}")
print(f"Imputation:        {IMPUTATION_METHOD}")
print(f"Mode:              {MODE}")
print(f"Load mode:         {LOAD_MODE}")
print(f"Input directory:   {DATA_ROOT}")
print(f"test.parquet used: no (reserved for inference)")
print(f"Output directory:  {OUTPUT_DIR}")
print(f"Reported GPU:      {GPU_NAME or 'not reported'}")
print(f"LightGBM device:   {DEVICE}")
if USE_GPU and DEVICE == "cpu":
    print("GPU requested but unavailable; using CPU.")
    for name, message in DEVICE_FAILURES.items():
        print(f"  {name}: {message}")
print("=" * 72)


## 2. Data Validation

Checks metadata stage, schema, required columns, and KNN donor contract when using the KNN pack.


In [ ]:
def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))

archive_meta = read_json(META_JSON)
dataset_meta = read_json(DATASET_METADATA_JSON)
raw_features = read_json(FEATURE_COLUMNS_JSON)

schema_probe = TABLE_PARQUETS[0]
parquet_file = pq.ParquetFile(schema_probe)
schema_columns = set(parquet_file.schema_arrow.names)
if dataset_meta.get("stage") != TRAINING_DATA_STAGE:
    raise ValueError(f"Metadata stage={dataset_meta.get('stage')} != {TRAINING_DATA_STAGE}")
if dataset_meta.get("s5p_2021_mode") != "ready":
    raise ValueError("Dataset must report S5P 2021 as ready")

missing_flag_column = None
if USE_PRECOMPUTED_KNN:
    missing_flag_column = "s2n_knn_imputed"
    if missing_flag_column not in schema_columns:
        raise ValueError("KNN pack must contain s2n_knn_imputed")
    knn_metadata = dataset_meta.get("knn", {})
    if (
        knn_metadata.get("n_neighbors") != 5
        or knn_metadata.get("weights") != "distance"
        or knn_metadata.get("donor_pool") != "train_years_<=_2022_and_s2n_available_eq_1"
    ):
        raise ValueError("KNN metadata does not match the train-only donor contract")

model_source_features = list(dict.fromkeys([
    *raw_features,
    *([missing_flag_column] if missing_flag_column else []),
]))

coverage_parts = [
    pd.read_parquet(path, columns=["cell_id", "label_date", "y_fire"])
    for path in TABLE_PARQUETS
]
coverage_probe = pd.concat(coverage_parts, ignore_index=True)
coverage_probe["label_date"] = pd.to_datetime(coverage_probe["label_date"])
if not USE_SPLIT:
    coverage_probe = coverage_probe.loc[coverage_probe["label_date"].dt.year.le(2024)].copy()
SOURCE_ROWS = int(len(coverage_probe))
SOURCE_CELLS = int(coverage_probe["cell_id"].nunique())
SOURCE_DAYS = int(coverage_probe["label_date"].nunique())
SOURCE_POSITIVES = int(coverage_probe["y_fire"].sum())
SOURCE_POSITIVE_RATE = SOURCE_POSITIVES / SOURCE_ROWS
YEAR_MIN = int(coverage_probe["label_date"].dt.year.min())
YEAR_MAX = int(coverage_probe["label_date"].dt.year.max())
del coverage_probe, coverage_parts
gc.collect()

if not USE_SPLIT and ALL_PARQUET.is_file():
    # Full-archive counts in metadata refer to all.parquet including 2025.
    full_rows = int(dataset_meta.get("n_rows", -1))
    full_pos = int(dataset_meta.get("n_pos", -1))
    print(f"Metadata full archive: rows={full_rows:,} positives={full_pos:,} (2025 not loaded for training)")

ID_COLUMNS = [
    "feature_end_date", "eo_asof_date", "label_date", "cell_id",
    "latitude", "longitude", "y_fire",
]
AGE_COLUMNS = ["s2n_lag_days", "s5n_lag_days"]
required_columns = set(ID_COLUMNS + AGE_COLUMNS + model_source_features)
missing_columns = sorted(required_columns - schema_columns)
if missing_columns:
    raise ValueError(f"Required columns are missing: {missing_columns}")

input_rows = []
for path in [META_JSON, DATASET_METADATA_JSON, FEATURE_COLUMNS_JSON, *TABLE_PARQUETS]:
    purpose = {
        META_JSON.name: "archive metadata",
        DATASET_METADATA_JSON.name: "stage / split metadata",
        FEATURE_COLUMNS_JSON.name: "feature allowlist",
        "train.parquet": "train 2019–2022",
        "val.parquet": "val 2023–2024",
        "all.parquet": "full table (filtered ≤2024)",
    }.get(path.name, path.name)
    input_rows.append({
        "file": path.name,
        "format": path.suffix.lstrip(".").upper(),
        "size_mb": path.stat().st_size / 1024**2,
        "purpose": purpose,
    })
if TEST_PARQUET.is_file():
    input_rows.append({
        "file": "test.parquet",
        "format": "PARQUET",
        "size_mb": TEST_PARQUET.stat().st_size / 1024**2,
        "purpose": "2025 — not loaded (inference only)",
    })
show(pd.DataFrame(input_rows).round({"size_mb": 3}))

quantity_overview = pd.DataFrame([
    {"measure": "Loaded rows (train work)", "quantity": SOURCE_ROWS},
    {"measure": "Columns on disk", "quantity": len(parquet_file.schema_arrow.names)},
    {"measure": "Allowlisted source features", "quantity": len(raw_features)},
    {"measure": "Additional KNN flag features", "quantity": int(USE_PRECOMPUTED_KNN)},
    {"measure": "Calendar days", "quantity": SOURCE_DAYS},
    {"measure": "Grid cells", "quantity": SOURCE_CELLS},
    {"measure": "Positive cell-days", "quantity": SOURCE_POSITIVES},
    {"measure": "Positive rate", "quantity": SOURCE_POSITIVE_RATE},
    {"measure": "Year range loaded", "quantity": f"{YEAR_MIN}–{YEAR_MAX}"},
])
show(quantity_overview)
if USE_PRECOMPUTED_KNN:
    print(f"Verified KNN donor pool: {dataset_meta['knn']['donor_pool']}")
print(f"Verified loaded coverage: {SOURCE_CELLS:,} cells × {SOURCE_DAYS:,} days = {SOURCE_ROWS:,} rows")


## 3. Preprocessing

Loads train+val (or `all` ≤2024), applies KNN pass-through or median-ready missing marks, optional fire-region cell filter.


In [ ]:
CONSTANT_FEATURES = {"s5n_s5p_aai_std", "s5n_s5p_co_std"}
base_features = [name for name in model_source_features if name not in CONSTANT_FEATURES]
load_columns = list(dict.fromkeys([*ID_COLUMNS, *AGE_COLUMNS, *model_source_features]))

def clean_source_table(frame: pd.DataFrame, raw_base_features: list[str]):
    frame = frame.copy()
    for name in ("feature_end_date", "eo_asof_date", "label_date"):
        frame[name] = pd.to_datetime(frame[name]).dt.normalize()
    frame = frame.sort_values(["cell_id", "label_date"]).reset_index(drop=True)
    cleanup_counts: dict[str, int] = {}

    if USE_PRECOMPUTED_KNN:
        frame[raw_base_features] = frame[raw_base_features].apply(pd.to_numeric, errors="raise").astype("float32")
        flag_values = set(frame[missing_flag_column].dropna().unique().tolist())
        if not flag_values.issubset({0.0, 1.0}):
            raise ValueError(f"{missing_flag_column} must be binary; found {sorted(flag_values)}")
        if not np.isfinite(frame[raw_base_features].to_numpy(dtype="float32", copy=False)).all():
            raise ValueError("Precomputed KNN values contain NaN or infinity")
        flagged = frame[missing_flag_column].eq(1)
        if not frame.loc[flagged, "s2n_available"].eq(0).all():
            raise ValueError("Every KNN-imputed S2 row must retain s2n_available=0")
        cleanup_counts["knn_imputed_rows_flagged"] = int(flagged.sum())
    else:
        s2_invalid = frame["s2n_available"].ne(1)
        s2_values = [name for name in raw_base_features if name.startswith("s2n_") and name != "s2n_available"]
        frame.loc[s2_invalid, s2_values] = np.nan
        frame.loc[s2_invalid, "s2n_available"] = 0.0
        cleanup_counts["sentinel2_rows_marked_missing"] = int(s2_invalid.sum())
        s5_invalid = frame["s5n_available"].ne(1)
        s5_values = [name for name in raw_base_features if name.startswith("s5n_") and name != "s5n_available"]
        frame.loc[s5_invalid, s5_values] = 0.0
        frame.loc[s5_invalid, "s5n_available"] = 0.0
        cleanup_counts["sentinel5p_rows_zeroed"] = int(s5_invalid.sum())
        frame[raw_base_features] = frame[raw_base_features].astype("float32")

    for name in ("swvl1_mean", "swvl2_mean", "soil_moisture_index", "swvl1_mean_7d"):
        cleanup_counts[f"{name}_negative_rows_clipped"] = int(frame[name].lt(0).sum())
        frame[name] = frame[name].clip(lower=0)
    frame["year"] = frame["label_date"].dt.year.astype("int16")
    return frame, cleanup_counts

started = time.time()

def resolve_fire_region_csv() -> Path:
    configured = (FIRE_REGION_CSV or "").strip()
    candidates: list[Path] = []
    if configured:
        candidates.append(Path(configured).expanduser())
    candidates.append(Path("/kaggle/input/datasets/lakshay654/firms-test/fire_analysis2.csv"))
    candidates.append(Path.cwd() / "fire_analysis2.csv")
    here = Path.cwd().resolve()
    candidates.extend([
        here / "Milestone 5" / "fire_analysis2.csv",
        here.parent / "Milestone 5" / "fire_analysis2.csv",
    ])
    if Path("/kaggle/input").is_dir():
        candidates.extend(Path("/kaggle/input").rglob("fire_analysis2.csv"))
    seen: set[Path] = set()
    for path in candidates:
        path = path.resolve()
        if path in seen:
            continue
        seen.add(path)
        if path.is_file():
            return path
    raise FileNotFoundError(
        f"fire_analysis2.csv not found for CELL_SUBSET={CELL_SUBSET}. "
        "Set FIRE_REGION_CSV or attach firms_test under /kaggle/input."
    )

def cells_for_subset(csv_path: Path, categories: list[str], available_cells: list) -> list:
    frame = pd.read_csv(csv_path)
    required = {"cell_id", "fire_region_category"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"fire_analysis2.csv missing columns: {sorted(missing)}")
    frame["cell_id"] = frame["cell_id"].astype(str)
    available = {str(cell) for cell in available_cells}
    counts = frame["fire_region_category"].value_counts(dropna=False).to_dict()
    print("fire_analysis2.csv category counts:", counts)
    allow = set(
        frame.loc[frame["fire_region_category"].isin(categories), "cell_id"].astype(str)
    )
    selected = [cell for cell in available_cells if str(cell) in allow]
    if not selected:
        raise ValueError(
            f"No cells matched CELL_SUBSET={CELL_SUBSET} categories={categories}"
        )
    unmatched_csv = sorted(allow - available)
    print(
        f"CELL_SUBSET={CELL_SUBSET}: categories={categories} "
        f"selected_cells={len(selected)} csv_only={len(unmatched_csv)}"
    )
    return selected

def load_training_tables(columns: list[str], cell_filter: list | None) -> pd.DataFrame:
    frames = []
    for path in TABLE_PARQUETS:
        if cell_filter is None:
            part = pd.read_parquet(path, columns=columns)
        else:
            part = pd.read_parquet(path, columns=columns, filters=[("cell_id", "in", cell_filter)])
        frames.append(part)
    frame = pd.concat(frames, ignore_index=True)
    frame["label_date"] = pd.to_datetime(frame["label_date"])
    # Never keep 2025 in training notebook work.
    frame = frame.loc[frame["label_date"].dt.year.le(2024)].copy()
    return frame

cell_parts = [pd.read_parquet(path, columns=["cell_id"]) for path in TABLE_PARQUETS]
cell_table = pd.concat(cell_parts, ignore_index=True)
archive_cells = sorted(cell_table["cell_id"].unique().tolist())
selected_cells = list(archive_cells)
FIRE_REGION_CSV_PATH = None
if MODE == "smoke":
    positions = np.linspace(0, len(selected_cells) - 1, min(96, len(selected_cells)), dtype=int)
    selected_cells = list(dict.fromkeys(selected_cells[position] for position in positions))
if CELL_SUBSET != "all":
    FIRE_REGION_CSV_PATH = resolve_fire_region_csv()
    selected_cells = cells_for_subset(
        FIRE_REGION_CSV_PATH,
        CELL_SUBSET_CATEGORIES[CELL_SUBSET],
        selected_cells,
    )
del cell_table, cell_parts
gc.collect()

SELECTED_TRAINING_CELLS = list(selected_cells)
CELL_SUBSET_INFO = {
    "cell_subset": CELL_SUBSET,
    "categories": CELL_SUBSET_CATEGORIES[CELL_SUBSET],
    "n_cells_selected": len(SELECTED_TRAINING_CELLS),
    "n_archive_cells": len(archive_cells),
    "fire_region_csv": str(FIRE_REGION_CSV_PATH) if FIRE_REGION_CSV_PATH else None,
}

cell_filter = None if (
    len(SELECTED_TRAINING_CELLS) == len(archive_cells) and CELL_SUBSET == "all" and MODE != "smoke"
) else SELECTED_TRAINING_CELLS
data = load_training_tables(load_columns, cell_filter)
data, cleanup = clean_source_table(data, base_features)
cells = int(data["cell_id"].nunique())
days = int(data["label_date"].nunique())
assert len(data) == cells * days
assert data.duplicated(["cell_id", "label_date"]).sum() == 0
assert sorted(data["y_fire"].unique().tolist()) == [0, 1]
assert (data["label_date"] - data["eo_asof_date"]).dt.days.eq(1).all()
assert (data["eo_asof_date"] - data["feature_end_date"]).dt.days.eq(5).all()
if cells != len(SELECTED_TRAINING_CELLS):
    raise ValueError(
        f"Loaded cells ({cells}) do not match selected subset ({len(SELECTED_TRAINING_CELLS)})"
    )
print(
    f"Loaded subset: cells={cells:,} days={days:,} rows={len(data):,} "
    f"positives={int(data['y_fire'].sum()):,} CELL_SUBSET={CELL_SUBSET} years={sorted(data['year'].unique().tolist())}"
)

yearly_quantity = data.groupby("year").agg(
    rows=("y_fire", "size"), positives=("y_fire", "sum"), days=("label_date", "nunique"),
)
yearly_quantity["positive_rate"] = yearly_quantity["positives"] / yearly_quantity["rows"]
yearly_quantity_display = yearly_quantity.copy()
yearly_quantity_display["positive_rate"] = yearly_quantity_display["positive_rate"].map(lambda value: f"{value:.3%}")
show(yearly_quantity_display)
print(json.dumps(cleanup, indent=2))
print(f"Loaded {len(data):,} rows in {time.time() - started:.1f}s")

plots_dir = OUTPUT_DIR / "plots"
figure, axis = plt.subplots(figsize=(9, 4.5))
axis.bar(yearly_quantity.index.astype(str), yearly_quantity["positives"], color="#c9472c")
axis.set(title="Positive wildfire cell-days by year (training load)", xlabel="Label year", ylabel="Positive rows")
for position, value in enumerate(yearly_quantity["positives"]):
    axis.text(position, value, f"{int(value):,}", ha="center", va="bottom", fontsize=9)
figure.tight_layout()
quantity_plot = plots_dir / "data_quantity_by_year.png"
figure.savefig(quantity_plot, dpi=150, bbox_inches="tight")
plt.close(figure)
show_image(quantity_plot)


## 4. Feature Table Creation

Builds calendar, weather history, and ignition/dryness features. No lagged fire-history / neighbor fire features.


In [ ]:
def rolling_matrix(values: np.ndarray, window: int, operation: str) -> np.ndarray:
    rolling = pd.DataFrame(values.T).rolling(window=window, min_periods=1)
    return getattr(rolling, operation)().to_numpy(dtype="float32").T

def build_features(frame: pd.DataFrame, raw_base_features: list[str]):
    cells = int(frame["cell_id"].nunique())
    days = int(frame["label_date"].nunique())

    # Calendar cycles.
    day_of_year = frame["eo_asof_date"].dt.dayofyear.to_numpy(dtype="float32")
    month = frame["eo_asof_date"].dt.month.to_numpy(dtype="float32")
    calendar = pd.DataFrame({
        "day_of_year_sin": np.sin(2 * np.pi * day_of_year / 365.25),
        "day_of_year_cos": np.cos(2 * np.pi * day_of_year / 365.25),
        "month_sin": np.sin(2 * np.pi * month / 12),
        "month_cos": np.cos(2 * np.pi * month / 12),
    }, index=frame.index).astype("float32")
    frame = pd.concat([frame, calendar], axis=1)

    # Weather physics and history ending at D-5.
    temperature_c = frame["t2m_mean"].to_numpy(dtype="float64") - 273.15
    dewpoint_c = frame["d2m_mean"].to_numpy(dtype="float64") - 273.15
    saturation = 0.6108 * np.exp(17.27 * temperature_c / np.maximum(temperature_c + 237.3, 1e-6))
    actual = 0.6108 * np.exp(17.27 * dewpoint_c / np.maximum(dewpoint_c + 237.3, 1e-6))
    vpd = np.maximum(saturation - actual, 0).astype("float32")
    weather = {
        "vpd_kpa": vpd,
        "vpd_wind_interaction": vpd * frame["wind_speed_mean"].to_numpy(dtype="float32"),
        "vpd_soil_deficit_interaction": vpd * (1 - np.clip(frame["soil_moisture_index"], 0, 1)),
        "heat_soil_deficit_interaction": (
            np.maximum(frame["t2m_max"].to_numpy(dtype="float32") - 273.15, 0)
            * (1 - np.clip(frame["swvl1_mean"], 0, 1))
        ),
        "wind_gust_ratio": frame["i10fg_max"].to_numpy(dtype="float32")
        / (frame["wind_speed_mean"].to_numpy(dtype="float32") + 0.1),
    }
    rolling_specs = {
        "t2m_max": ("max",), "rh_mean": ("min",), "tp_sum_mm": ("sum",),
        "wind_speed_mean": ("max",), "i10fg_max": ("max",),
        "swvl1_mean": ("mean",), "vpd_kpa": ("max", "mean"),
    }
    arrays = {
        name: (weather[name] if name in weather else frame[name].to_numpy(dtype="float32")).reshape(cells, days)
        for name in rolling_specs
    }
    for name, operations in rolling_specs.items():
        for window in (14, 30):
            for operation in operations:
                weather[f"{name}_{operation}_{window}d"] = rolling_matrix(
                    arrays[name], window, operation
                ).reshape(-1)
    temperature = frame["t2m_max"].to_numpy(dtype="float32").reshape(cells, days)
    soil = frame["swvl1_mean"].to_numpy(dtype="float32").reshape(cells, days)
    weather["t2m_max_anomaly_30d"] = (temperature - rolling_matrix(temperature, 30, "mean")).reshape(-1)
    weather["swvl1_anomaly_30d"] = (soil - rolling_matrix(soil, 30, "mean")).reshape(-1)
    weather_frame = pd.DataFrame(weather, index=frame.index).astype("float32")
    frame = pd.concat([frame, weather_frame], axis=1)

    # Environmental ignition / dryness context only (no lagged fire labels).
    # Cell/neighbor/upwind fire-history features are omitted: they encode spread /
    # persistence and dominate next-day environmental risk scoring.
    wind = frame["wind_speed_mean"].to_numpy(dtype="float32")
    vpd = frame["vpd_kpa"].to_numpy(dtype="float32")
    soil_deficit = 1 - np.clip(frame["soil_moisture_index"].to_numpy(dtype="float32"), 0, 1)
    vegetation = np.clip(
        frame["cvh_mean"].to_numpy(dtype="float32") + frame["cvl_mean"].to_numpy(dtype="float32"), 0, 1
    )
    context = {
        "ignition_dry_windy_index": vpd * wind * soil_deficit,
        "fuel_dryness_index": vpd * soil_deficit * vegetation,
        "vpd_short_long_trend": frame["vpd_kpa_mean_14d"] - frame["vpd_kpa_mean_30d"],
    }
    context_frame = pd.DataFrame(context, index=frame.index).astype("float32")
    frame = pd.concat([frame, context_frame], axis=1)

    # The locked contract excludes a redundant source-availability flag.
    selected_base = [name for name in raw_base_features if name != "s5n_available"]
    feature_columns = list(dict.fromkeys([
        *selected_base,
        "latitude", "longitude",
        *calendar.columns,
        *weather_frame.columns,
        *context_frame.columns,
    ]))
    groups = {
        "source_after_constant_removal": len(raw_base_features),
        "source_used_by_model": len(selected_base),
        "geographic": 2,
        "calendar": len(calendar.columns),
        "weather_and_interactions": len(weather_frame.columns),
        "dryness_and_ignition_context": len(context_frame.columns),
        "total": len(feature_columns),
    }
    return frame, feature_columns, groups


In [ ]:
started = time.time()
data, feature_columns, feature_groups = build_features(data, base_features)
EXPECTED_FEATURE_COUNT = 93 if USE_PRECOMPUTED_KNN else 92
if len(feature_columns) != EXPECTED_FEATURE_COUNT:
    raise ValueError(f"Expected {EXPECTED_FEATURE_COUNT} features, found {len(feature_columns)}")
values = data[feature_columns].to_numpy(dtype="float32", copy=False)
if np.isinf(values).any():
    raise ValueError("An infinite engineered feature was found")
if USE_PRECOMPUTED_KNN and np.isnan(values).any():
    raise ValueError("KNN feature table contains NaN")
missing_feature_values = int(np.isnan(values).sum())

show(pd.DataFrame([
    {"feature_group": name.replace("_", " ").title(), "count": count}
    for name, count in feature_groups.items()
]))
print(f"Created the {EXPECTED_FEATURE_COUNT}-feature table in {time.time() - started:.1f}s")
print(f"Values awaiting pipeline imputation: {missing_feature_values:,}")


## 5. Train / Val / Test Split

Train ≤2022, validation 2023, calibration 2024. `test.parquet` (2025) stays for inference only.


In [ ]:
training = data.loc[data["year"].le(2022)].copy()
validation = data.loc[data["year"].eq(2023)].copy()
calibration = data.loc[data["year"].eq(2024)].copy()
if training.empty or validation.empty or calibration.empty:
    raise ValueError("Expected years 2019–2022 train, 2023 validation, 2024 calibration in the loaded tables")
del data, values
gc.collect()

split_quantity = pd.DataFrame([
    {
        "split": name,
        "label_years": (
            f"{part['year'].min()}–{part['year'].max()}"
            if part["year"].nunique() > 1 else str(int(part["year"].iloc[0]))
        ),
        "rows": len(part),
        "positives": int(part["y_fire"].sum()),
        "positive_rate": float(part["y_fire"].mean()),
        "purpose": purpose,
    }
    for name, part, purpose in [
        ("training", training, "fit classifier, ranker, preprocessing"),
        ("validation", validation, "Optuna + validation metrics"),
        ("calibration", calibration, "probability calibrator only"),
    ]
])
split_quantity_display = split_quantity.copy()
split_quantity_display["rows"] = split_quantity_display["rows"].map(lambda value: f"{value:,}")
split_quantity_display["positives"] = split_quantity_display["positives"].map(lambda value: f"{value:,}")
split_quantity_display["positive_rate"] = split_quantity_display["positive_rate"].map(lambda value: f"{value:.3%}")
show(split_quantity_display)
print("test.parquet / 2025 is not loaded here — use the inference notebook for 2025.")


## 6. Training Pipeline

Defines classifier + ranker pipelines, then Optuna tunes on 2023 (best params for the next step).


In [ ]:
def make_preprocessor():
    if USE_PRECOMPUTED_KNN:
        return FunctionTransformer(validate=False, feature_names_out="one-to-one")
    return SimpleImputer(strategy="median")

# Locked defaults (pre-Optuna). Optuna overwrites CLASSIFIER_PARAMS / RANKER_PARAMS when enabled.
DEFAULT_CLASSIFIER_PARAMS = {
    "objective": "binary",
    "n_estimators": 248,
    "learning_rate": 0.025,
    "num_leaves": 31,
    "min_child_samples": 75,
    "colsample_bytree": 0.90,
    "subsample": 0.90,
    "subsample_freq": 1,
    "reg_alpha": 0.2,
    "reg_lambda": 3.0,
    "scale_pos_weight": 1.0,
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
    "feature_pre_filter": False,
    "device_type": DEVICE,
}
DEFAULT_RANKER_PARAMS = {
    "objective": "lambdarank",
    "n_estimators": 221,
    "learning_rate": 0.03,
    "num_leaves": 63,
    "min_child_samples": 100,
    "colsample_bytree": 0.85,
    "subsample": 0.85,
    "subsample_freq": 1,
    "reg_lambda": 8.0,
    "lambdarank_truncation_level": 100,
    "label_gain": [0, 1],
    "random_state": SEED,
    "n_jobs": -1,
    "verbosity": -1,
    "feature_pre_filter": False,
    "device_type": DEVICE,
}
CLASSIFIER_PARAMS = dict(DEFAULT_CLASSIFIER_PARAMS)
RANKER_PARAMS = dict(DEFAULT_RANKER_PARAMS)

def build_classifier_pipeline(params: dict[str, Any]) -> Pipeline:
    return Pipeline([
        ("feature_preprocessor", make_preprocessor()),
        ("fire_probability_model", lgb.LGBMClassifier(**params)),
    ])

def build_ranker_pipeline(params: dict[str, Any]) -> Pipeline:
    return Pipeline([
        ("feature_preprocessor", make_preprocessor()),
        ("daily_priority_model", lgb.LGBMRanker(**params)),
    ])

classifier_pipeline = build_classifier_pipeline(CLASSIFIER_PARAMS)
ranker_pipeline = build_ranker_pipeline(RANKER_PARAMS)

print("Classifier pipeline")
show(classifier_pipeline)
print("Ranker pipeline")
show(ranker_pipeline)


In [ ]:
# Toggle: env WILDFIRE_OPTUNA=0|1, WILDFIRE_OPTUNA_TRIALS=N, WILDFIRE_OPTUNA_RANKER=0|1
ENABLE_OPTUNA = env_get("WILDFIRE_OPTUNA", "CHAMPION_OPTUNA", default="1").lower() not in {"0", "false", "no"}
TUNE_RANKER = env_get("WILDFIRE_OPTUNA_RANKER", "CHAMPION_OPTUNA_RANKER", default="0").lower() not in {"0", "false", "no"}
OPTUNA_TRIALS = int(env_get(
    "WILDFIRE_OPTUNA_TRIALS", "CHAMPION_OPTUNA_TRIALS",
    default="8" if MODE == "smoke" else "25",
))
OPTUNA_TIMEOUT_SEC = None if MODE == "smoke" else int(env_get("WILDFIRE_OPTUNA_TIMEOUT", "CHAMPION_OPTUNA_TIMEOUT", default="1800"))
EARLY_STOPPING_ROUNDS = 40 if MODE == "smoke" else 60

try:
    import optuna
    from optuna.samplers import TPESampler
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError as error:
    raise ImportError(
        "Optuna is required for this cell. Install with: pip install optuna"
    ) from error

X_train = training[feature_columns]
y_train = training["y_fire"].to_numpy(dtype="int8")
X_valid = validation[feature_columns]
y_valid = validation["y_fire"].to_numpy(dtype="int8")

# Preprocess once outside trials (imputer / identity), then tune the booster only.
_pre = make_preprocessor()
X_train_m = _pre.fit_transform(X_train)
X_valid_m = _pre.transform(X_valid)

def _pr_auc(y_true: np.ndarray, y_score: np.ndarray) -> float:
    return float(average_precision_score(y_true, y_score))

def _fit_classifier_booster(params: dict[str, Any], n_estimators: int) -> tuple[lgb.LGBMClassifier, int, float]:
    model = lgb.LGBMClassifier(**{**params, "n_estimators": n_estimators})
    model.fit(
        X_train_m,
        y_train,
        eval_set=[(X_valid_m, y_valid)],
        eval_metric="average_precision",
        callbacks=[
            lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
            lgb.log_evaluation(period=0),
        ],
    )
    best_iteration = int(getattr(model, "best_iteration_", None) or model.n_estimators_)
    score = _pr_auc(y_valid, model.predict_proba(X_valid_m)[:, 1])
    return model, best_iteration, score

optuna_summary: dict[str, Any] = {
    "enabled": ENABLE_OPTUNA,
    "trials_requested": OPTUNA_TRIALS,
    "tune_ranker": TUNE_RANKER,
    "objective": "validation_pr_auc_2023",
}

if not ENABLE_OPTUNA:
    print("Optuna disabled — using locked DEFAULT_* params.")
    CLASSIFIER_PARAMS = dict(DEFAULT_CLASSIFIER_PARAMS)
    RANKER_PARAMS = dict(DEFAULT_RANKER_PARAMS)
else:
    print("=" * 72)
    print(f"OPTUNA classifier search: {OPTUNA_TRIALS} trials | early_stopping={EARLY_STOPPING_ROUNDS}")
    print("=" * 72)

    def classifier_objective(trial: optuna.Trial) -> float:
        # Bias search toward regularized / lower-capacity models (anti-overfit).
        params = {
            "objective": "binary",
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 15, 63),
            "min_child_samples": trial.suggest_int("min_child_samples", 50, 250),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "subsample_freq": 1,
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 20.0, log=True),
            "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, 8.0),
            "random_state": SEED,
            "n_jobs": -1,
            "verbosity": -1,
            "feature_pre_filter": False,
            "device_type": DEVICE,
        }
        n_estimators = trial.suggest_int("n_estimators_cap", 150, 500)
        model, best_iteration, score = _fit_classifier_booster(params, n_estimators)
        trial.set_user_attr("best_iteration", best_iteration)
        trial.set_user_attr("train_pr_auc", _pr_auc(y_train, model.predict_proba(X_train_m)[:, 1]))
        return score

    sampler = TPESampler(seed=SEED)
    classifier_study = optuna.create_study(direction="maximize", sampler=sampler, study_name="classifier_pr_auc")
    classifier_study.optimize(
        classifier_objective,
        n_trials=OPTUNA_TRIALS,
        timeout=OPTUNA_TIMEOUT_SEC,
        show_progress_bar=True,
    )

    best = classifier_study.best_trial
    CLASSIFIER_PARAMS = {
        "objective": "binary",
        "n_estimators": int(best.user_attrs["best_iteration"]),
        "learning_rate": best.params["learning_rate"],
        "num_leaves": best.params["num_leaves"],
        "min_child_samples": best.params["min_child_samples"],
        "colsample_bytree": best.params["colsample_bytree"],
        "subsample": best.params["subsample"],
        "subsample_freq": 1,
        "reg_alpha": best.params["reg_alpha"],
        "reg_lambda": best.params["reg_lambda"],
        "scale_pos_weight": best.params["scale_pos_weight"],
        "random_state": SEED,
        "n_jobs": -1,
        "verbosity": -1,
        "feature_pre_filter": False,
        "device_type": DEVICE,
    }
    optuna_summary.update({
        "classifier_best_pr_auc": float(best.value),
        "classifier_best_params": dict(CLASSIFIER_PARAMS),
        "classifier_train_pr_auc": float(best.user_attrs.get("train_pr_auc", float("nan"))),
        "classifier_trials_done": len(classifier_study.trials),
    })
    gap = optuna_summary["classifier_train_pr_auc"] - optuna_summary["classifier_best_pr_auc"]
    print(f"Best 2023 PR-AUC: {best.value:.6f} | train PR-AUC: {optuna_summary['classifier_train_pr_auc']:.6f} | gap: {gap:.6f}")
    show(pd.DataFrame([CLASSIFIER_PARAMS]).T.rename(columns={0: "value"}))

    if TUNE_RANKER:
        print("=" * 72)
        print("OPTUNA ranker search (proxy: daily mean average precision on 2023)")
        print("=" * 72)
        rank_train = training.sort_values(["label_date", "cell_id"])
        rank_valid = validation.sort_values(["label_date", "cell_id"])
        train_groups = rank_train.groupby("label_date", sort=False).size().to_numpy(dtype="int32")
        X_rt = _pre.transform(rank_train[feature_columns])
        y_rt = rank_train["y_fire"].to_numpy(dtype="int8")
        X_rv = _pre.transform(rank_valid[feature_columns])
        y_rv = rank_valid["y_fire"].to_numpy(dtype="int8")
        valid_dates = rank_valid["label_date"].to_numpy()

        def _daily_map(y_true: np.ndarray, scores: np.ndarray, dates: np.ndarray) -> float:
            frame = pd.DataFrame({"y": y_true, "s": scores, "d": dates})
            maps = []
            for _, group in frame.groupby("d", sort=False):
                if group["y"].sum() <= 0:
                    continue
                maps.append(average_precision_score(group["y"], group["s"]))
            return float(np.mean(maps)) if maps else 0.0

        def ranker_objective(trial: optuna.Trial) -> float:
            params = {
                "objective": "lambdarank",
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.08, log=True),
                "num_leaves": trial.suggest_int("num_leaves", 15, 63),
                "min_child_samples": trial.suggest_int("min_child_samples", 50, 250),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "subsample_freq": 1,
                "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 30.0, log=True),
                "lambdarank_truncation_level": trial.suggest_int("lambdarank_truncation_level", 50, 150),
                "label_gain": [0, 1],
                "random_state": SEED,
                "n_jobs": -1,
                "verbosity": -1,
                "feature_pre_filter": False,
                "device_type": DEVICE,
            }
            n_estimators = trial.suggest_int("n_estimators_cap", 150, 400)
            model = lgb.LGBMRanker(**{**params, "n_estimators": n_estimators})
            model.fit(
                X_rt,
                y_rt,
                group=train_groups,
                eval_set=[(X_rv, y_rv)],
                eval_group=[rank_valid.groupby("label_date", sort=False).size().to_numpy(dtype="int32")],
                eval_metric="map",
                callbacks=[
                    lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                    lgb.log_evaluation(period=0),
                ],
            )
            best_iteration = int(getattr(model, "best_iteration_", None) or model.n_estimators_)
            score = _daily_map(y_rv, model.predict(X_rv), valid_dates)
            trial.set_user_attr("best_iteration", best_iteration)
            return score

        ranker_study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED + 1), study_name="ranker_map")
        ranker_study.optimize(
            ranker_objective,
            n_trials=max(5, OPTUNA_TRIALS // 2),
            timeout=OPTUNA_TIMEOUT_SEC,
            show_progress_bar=True,
        )
        rb = ranker_study.best_trial
        RANKER_PARAMS = {
            "objective": "lambdarank",
            "n_estimators": int(rb.user_attrs["best_iteration"]),
            "learning_rate": rb.params["learning_rate"],
            "num_leaves": rb.params["num_leaves"],
            "min_child_samples": rb.params["min_child_samples"],
            "colsample_bytree": rb.params["colsample_bytree"],
            "subsample": rb.params["subsample"],
            "subsample_freq": 1,
            "reg_lambda": rb.params["reg_lambda"],
            "lambdarank_truncation_level": rb.params["lambdarank_truncation_level"],
            "label_gain": [0, 1],
            "random_state": SEED,
            "n_jobs": -1,
            "verbosity": -1,
            "feature_pre_filter": False,
            "device_type": DEVICE,
        }
        optuna_summary["ranker_best_daily_map"] = float(rb.value)
        optuna_summary["ranker_best_params"] = dict(RANKER_PARAMS)
    else:
        RANKER_PARAMS = dict(DEFAULT_RANKER_PARAMS)
        print("Ranker Optuna skipped (WILDFIRE_OPTUNA_RANKER=0). Using defaults.")

# Rebuild pipelines from the selected params before the fit cell.
classifier_pipeline = build_classifier_pipeline(CLASSIFIER_PARAMS)
ranker_pipeline = build_ranker_pipeline(RANKER_PARAMS)

def _json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): _json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(v) for v in value]
    if isinstance(value, (np.floating, float)):
        return float(value)
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if value is None or isinstance(value, str):
        return value
    return str(value)

(OUTPUT_DIR / "metrics").mkdir(parents=True, exist_ok=True)
optuna_path = OUTPUT_DIR / "metrics" / "optuna_summary.json"
optuna_path.write_text(json.dumps(_json_ready(optuna_summary), indent=2), encoding="utf-8")
print(f"Wrote {optuna_path}")
show(pd.DataFrame([{
    "optuna_enabled": ENABLE_OPTUNA,
    "classifier_n_estimators": CLASSIFIER_PARAMS["n_estimators"],
    "classifier_num_leaves": CLASSIFIER_PARAMS["num_leaves"],
    "classifier_min_child_samples": CLASSIFIER_PARAMS["min_child_samples"],
    "classifier_reg_lambda": CLASSIFIER_PARAMS["reg_lambda"],
    "val_pr_auc_best": optuna_summary.get("classifier_best_pr_auc"),
    "train_val_pr_auc_gap": (
        None if "classifier_best_pr_auc" not in optuna_summary
        else optuna_summary["classifier_train_pr_auc"] - optuna_summary["classifier_best_pr_auc"]
    ),
}]))


## 7. Training of Model

Fits classifier and ranker on 2019–2022 using the selected hyperparameters.


In [ ]:
started = time.time()
classifier_pipeline.fit(training[feature_columns], training["y_fire"])
classifier_seconds = time.time() - started

rank_training = training.sort_values(["label_date", "cell_id"])
daily_group_sizes = rank_training.groupby("label_date", sort=False).size().to_numpy(dtype="int32")
started = time.time()
ranker_pipeline.fit(
    rank_training[feature_columns],
    rank_training["y_fire"],
    daily_priority_model__group=daily_group_sizes,
)
ranker_seconds = time.time() - started

training_summary = pd.DataFrame([
    {"component": "classifier pipeline", "device": DEVICE, "fit_seconds": classifier_seconds, "training_rows": len(training)},
    {"component": "ranker pipeline", "device": DEVICE, "fit_seconds": ranker_seconds, "training_rows": len(rank_training)},
])
training_summary_display = training_summary.copy()
training_summary_display["fit_seconds"] = training_summary_display["fit_seconds"].map(lambda value: f"{value:.1f}")
training_summary_display["training_rows"] = training_summary_display["training_rows"].map(lambda value: f"{value:,}")
show(training_summary_display)
print("Active classifier params:")
show(pd.DataFrame([CLASSIFIER_PARAMS]).T.rename(columns={0: "value"}))


## 8. Validation

Scores **2023** (full year + last 14 days), then fits the probability calibrator on **2024**. Metrics: PR-AUC, ROC-AUC, F1@0.5, F1/precision/recall@top-25.


In [ ]:
def within_day_percentile(score: np.ndarray, dates: pd.Series) -> np.ndarray:
    table = pd.DataFrame({"date": pd.to_datetime(dates).to_numpy(), "score": score, "position": np.arange(len(score))})
    table["percentile"] = table.groupby("date", sort=False)["score"].rank(method="average", pct=True)
    return table.sort_values("position")["percentile"].to_numpy(dtype=float)

def top_k_alert_metrics(frame: pd.DataFrame, k: int, score_column: str) -> dict[str, float | int]:
    alerts = (
        frame.sort_values(["label_date", score_column], ascending=[True, False])
        .groupby("label_date", group_keys=False)
        .head(k)
    )
    y_alert = alerts["y_fire"].to_numpy(dtype="int8")
    captured = int(y_alert.sum())
    positives = int(frame["y_fire"].sum())
    days = max(int(frame["label_date"].nunique()), 1)
    alert_keys = set(zip(alerts["label_date"].astype(str), alerts["cell_id"].astype(str)))
    pred = np.array([
        1 if (str(d), str(c)) in alert_keys else 0
        for d, c in zip(frame["label_date"], frame["cell_id"])
    ], dtype="int8")
    y_true = frame["y_fire"].to_numpy(dtype="int8")
    return {
        "alerts": len(alerts),
        "precision_at_k": captured / max(len(alerts), 1),
        "recall_at_k": captured / max(positives, 1),
        "f1_at_k": float(f1_score(y_true, pred, zero_division=0)),
        "false_alerts_per_day": (len(alerts) - captured) / days,
    }

def score_frame(frame: pd.DataFrame) -> pd.DataFrame:
    raw_probability = classifier_pipeline.predict_proba(frame[feature_columns])[:, 1]
    raw_clipped = np.clip(raw_probability, 1e-7, 1 - 1e-7)
    calibrated_probability = probability_calibrator.predict_proba(
        np.log(raw_clipped / (1 - raw_clipped)).reshape(-1, 1)
    )[:, 1]
    rank_sorted = frame.sort_values(["label_date", "cell_id"])
    rank_score_sorted = ranker_pipeline.predict(rank_sorted[feature_columns])
    rank_predictions = pd.Series(rank_score_sorted, index=rank_sorted.index).reindex(frame.index).to_numpy(dtype=float)
    classifier_percentile = within_day_percentile(raw_probability, frame["label_date"])
    ranker_percentile = within_day_percentile(rank_predictions, frame["label_date"])
    alert_score = 0.50 * classifier_percentile + 0.50 * ranker_percentile
    scored = frame[["feature_end_date", "eo_asof_date", "label_date", "cell_id", "latitude", "longitude", "y_fire"]].copy()
    scored["p_fire_raw"] = raw_probability.astype("float32")
    scored["p_fire"] = calibrated_probability.astype("float32")
    scored["rank_score"] = rank_predictions.astype("float32")
    scored["alert_score"] = alert_score.astype("float32")
    return scored

def compute_metrics(scored: pd.DataFrame, label: str) -> dict[str, float | int | str]:
    y_true = scored["y_fire"].to_numpy(dtype="int8")
    p = scored["p_fire"].to_numpy(dtype=float)
    pred_05 = (p >= 0.5).astype("int8")
    alert_25 = top_k_alert_metrics(scored, 25, "alert_score")
    alert_50 = top_k_alert_metrics(scored, 50, "alert_score")
    return {
        "slice": label,
        "rows": len(scored),
        "positives": int(y_true.sum()),
        "prevalence": float(y_true.mean()) if len(scored) else 0.0,
        "pr_auc": float(average_precision_score(y_true, p)) if y_true.sum() > 0 else float("nan"),
        "roc_auc": float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) > 1 else float("nan"),
        "f1_at_0_5": float(f1_score(y_true, pred_05, zero_division=0)),
        "precision_at_0_5": float(precision_score(y_true, pred_05, zero_division=0)),
        "recall_at_0_5": float(recall_score(y_true, pred_05, zero_division=0)),
        "brier": float(brier_score_loss(y_true, p)),
        "log_loss": float(log_loss(y_true, np.clip(p, 1e-7, 1 - 1e-7), labels=[0, 1])),
        "f1_at_25": alert_25["f1_at_k"],
        "precision_at_25": alert_25["precision_at_k"],
        "recall_at_25": alert_25["recall_at_k"],
        "false_alerts_per_day_at_25": alert_25["false_alerts_per_day"],
        "recall_at_50": alert_50["recall_at_k"],
    }

def format_metrics_table(rows: list[dict[str, float | int | str]]) -> pd.DataFrame:
    display = pd.DataFrame(rows)
    pct_cols = [
        "prevalence", "f1_at_0_5", "precision_at_0_5", "recall_at_0_5",
        "f1_at_25", "precision_at_25", "recall_at_25", "recall_at_50",
    ]
    float_cols = ["pr_auc", "roc_auc", "brier", "log_loss", "false_alerts_per_day_at_25"]
    for name in pct_cols:
        if name in display:
            display[name] = display[name].map(lambda value: f"{value:.3%}" if pd.notna(value) else "—")
    for name in float_cols:
        if name in display:
            display[name] = display[name].map(lambda value: f"{value:.6f}" if pd.notna(value) else "—")
    return display

# Fit probability calibrator on 2024 only (after model fit).
raw_calibration_probability = classifier_pipeline.predict_proba(calibration[feature_columns])[:, 1]
clipped = np.clip(raw_calibration_probability, 1e-7, 1 - 1e-7)
calibration_logit = np.log(clipped / (1 - clipped)).reshape(-1, 1)
probability_calibrator = LogisticRegression(
    C=1e6, solver="lbfgs", max_iter=1000, random_state=SEED
).fit(calibration_logit, calibration["y_fire"])
print(f"Calibrator fitted on 2024 rows={len(calibration):,}")

# Full 2023 validation + last-14-day sample.
validation_scored = score_frame(validation)
last_dates = sorted(validation_scored["label_date"].dropna().unique())[-14:]
validation_last14 = validation_scored.loc[validation_scored["label_date"].isin(last_dates)].copy()

validation_metrics = compute_metrics(validation_scored, "2023 full year")
validation_last14_metrics = compute_metrics(validation_last14, "2023 last 14 days")
train_raw = classifier_pipeline.predict_proba(training[feature_columns])[:, 1]
train_pr_auc = float(average_precision_score(training["y_fire"], train_raw))
validation_metrics["train_pr_auc"] = train_pr_auc
validation_metrics["train_val_pr_auc_gap"] = train_pr_auc - float(validation_metrics["pr_auc"])

print("Metric definitions:")
show(pd.DataFrame([
    {"metric": "PR-AUC", "meaning": "Average precision; ranking quality with rare positives"},
    {"metric": "ROC-AUC", "meaning": "Separation of fire vs non-fire scores"},
    {"metric": "F1 @ 0.5", "meaning": "F1 when p_fire >= 0.5"},
    {"metric": "F1 / Precision / Recall @ 25", "meaning": "Daily top-25 cells by alert_score treated as alerts"},
    {"metric": "False alerts/day @ 25", "meaning": "Average non-fire cells inside daily top-25"},
]))

print("Validation metrics")
show(format_metrics_table([validation_metrics, validation_last14_metrics]))
if validation_metrics["train_val_pr_auc_gap"] > 0.15:
    print(
        "Warning: large train–val PR-AUC gap. Prefer stronger regularization "
        "or more Optuna trials biased toward min_child_samples / reg_lambda."
    )

metrics_dir = OUTPUT_DIR / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)

def json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (Path, pd.Timestamp, datetime)):
        return str(value)
    if value is None or isinstance(value, str):
        return value
    return str(value)

def write_json(payload: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(json_ready(payload), indent=2, sort_keys=True) + "\n", encoding="utf-8")

write_json(validation_metrics, metrics_dir / "validation_2023.json")
write_json(validation_last14_metrics, metrics_dir / "validation_2023_last14d.json")

# Plots on 2023 validation (not 2025).
y_plot = validation_scored["y_fire"].to_numpy(dtype="int8")
p_plot = validation_scored["p_fire"].to_numpy(dtype=float)
observed, predicted = calibration_curve(y_plot, p_plot, n_bins=10, strategy="quantile")
precision, recall, _ = precision_recall_curve(y_plot, p_plot)
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(predicted, observed, marker="o", label="model")
axes[0].plot([0, 1], [0, 1], "--", color="black", label="ideal")
axes[0].set(title="2023 probability reliability", xlabel="Predicted probability", ylabel="Observed rate")
axes[0].legend()
axes[1].plot(recall, precision, color="#c9472c")
axes[1].axhline(y_plot.mean(), linestyle="--", color="gray", label="prevalence")
axes[1].set(title="2023 precision–recall", xlabel="Recall", ylabel="Precision")
axes[1].legend()
figure.tight_layout()
calibration_plot = plots_dir / "validation_calibration_and_pr.png"
figure.savefig(calibration_plot, dpi=160, bbox_inches="tight")
plt.close(figure)
show_image(calibration_plot)

peak_day = validation_scored.groupby("label_date")["p_fire"].max().idxmax()
day = validation_scored.loc[validation_scored["label_date"].eq(peak_day)]
figure, axis = plt.subplots(figsize=(7, 8))
points = axis.scatter(
    day["longitude"], day["latitude"], c=day["p_fire"], cmap="YlOrRd",
    vmin=0, vmax=max(float(day["p_fire"].quantile(0.99)), 1e-5), s=28,
)
positives = day.loc[day["y_fire"].eq(1)]
if len(positives):
    axis.scatter(positives["longitude"], positives["latitude"], marker="x", color="black", s=38, label="observed positive")
    axis.legend()
axis.set(title=f"2023 peak-risk day — {pd.Timestamp(peak_day).date()}", xlabel="Longitude", ylabel="Latitude")
figure.colorbar(points, ax=axis, label="Calibrated probability")
figure.tight_layout()
risk_map = plots_dir / "validation_peak_day_risk_map.png"
figure.savefig(risk_map, dpi=160, bbox_inches="tight")
plt.close(figure)
show_image(risk_map)


## 9. Feature Importance

TreeSHAP contributions from the fitted classifier on a 2023 sample.


In [ ]:
explain_dir = OUTPUT_DIR / "explainability"
explain_dir.mkdir(parents=True, exist_ok=True)
sample = validation.sample(min(2000, len(validation)), random_state=SEED)
feature_preprocessor = classifier_pipeline.named_steps["feature_preprocessor"]
classifier_model = classifier_pipeline.named_steps["fire_probability_model"]
matrix = feature_preprocessor.transform(sample[feature_columns])
contributions = classifier_model.booster_.predict(matrix, pred_contrib=True)
shap_values = np.asarray(contributions)[:, :-1]
importance = pd.DataFrame({
    "feature": feature_columns,
    "mean_absolute_contribution": np.abs(shap_values).mean(axis=0),
    "gain_importance": classifier_model.booster_.feature_importance(importance_type="gain"),
}).sort_values("mean_absolute_contribution", ascending=False)
importance.to_csv(explain_dir / "feature_explanations.csv", index=False)
show(importance.head(20))

top = importance.head(20).sort_values("mean_absolute_contribution")
figure, axis = plt.subplots(figsize=(9, 7))
axis.barh(top["feature"], top["mean_absolute_contribution"], color="#c9472c")
axis.set(title="Most influential classifier features (2023 sample)", xlabel="Mean absolute TreeSHAP contribution")
figure.tight_layout()
importance_plot = explain_dir / "feature_explanations.png"
figure.savefig(importance_plot, dpi=160, bbox_inches="tight")
plt.close(figure)
show_image(importance_plot)


## 10. Output Submission

Writes `models/wildfire_model.joblib` and metrics under `/kaggle/working/...`. Save Version and attach this output to the inference notebook.


In [ ]:
def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

models_dir = OUTPUT_DIR / "models"
models_dir.mkdir(parents=True, exist_ok=True)
model_path = models_dir / "wildfire_model.joblib"
classifier_weights_path = models_dir / "classifier_weights.txt"
ranker_weights_path = models_dir / "ranker_weights.txt"

archive_last_label_date = str(calibration["label_date"].max().date())
model_artifact = {
    "artifact_name": "wildfire",
    "data_source": DATA_SOURCE,
    "source_stage": TRAINING_DATA_STAGE,
    "cell_subset": CELL_SUBSET,
    "imputation_method": IMPUTATION_METHOD,
    "missing_flag_column": missing_flag_column,
    "classifier_pipeline": classifier_pipeline,
    "ranker_pipeline": ranker_pipeline,
    "probability_calibrator": probability_calibrator,
    "feature_columns": feature_columns,
    "classifier_params": CLASSIFIER_PARAMS,
    "ranker_params": RANKER_PARAMS,
    "optuna": {
        "enabled": bool(globals().get("ENABLE_OPTUNA", False)),
        "summary": globals().get("optuna_summary"),
    },
    "raw_feature_columns": model_source_features,
    "base_features": base_features,
    "source_columns": load_columns,
    "classifier_weight": 0.50,
    "ranker_weight": 0.50,
    "data_contract": {
        "label_offset_from_eo_asof_days": 1,
        "eo_asof_offset_from_feature_end_days": 5,
        "expected_grid_cells": len(SELECTED_TRAINING_CELLS),
        "archive_source_cells": SOURCE_CELLS,
        "cell_subset": CELL_SUBSET,
        "cell_subset_categories": CELL_SUBSET_CATEGORIES[CELL_SUBSET],
        "selected_cell_ids": SELECTED_TRAINING_CELLS,
        "fire_region_csv": CELL_SUBSET_INFO.get("fire_region_csv"),
        "archive_last_label_date": archive_last_label_date,
        "training_years": [2019, 2020, 2021, 2022],
        "validation_year": 2023,
        "calibration_year": 2024,
        "test_year_reserved_for_inference": 2025,
        "load_mode": LOAD_MODE,
    },
}
cell_subset_path = models_dir / "selected_cells.json"
write_json({
    "cell_subset": CELL_SUBSET,
    "categories": CELL_SUBSET_CATEGORIES[CELL_SUBSET],
    "n_cells_selected": len(SELECTED_TRAINING_CELLS),
    "cell_ids": SELECTED_TRAINING_CELLS,
    "fire_region_csv": CELL_SUBSET_INFO.get("fire_region_csv"),
}, cell_subset_path)

joblib.dump(model_artifact, model_path)
classifier_pipeline.named_steps["fire_probability_model"].booster_.save_model(str(classifier_weights_path))
ranker_pipeline.named_steps["daily_priority_model"].booster_.save_model(str(ranker_weights_path))

feature_contract_path = OUTPUT_DIR / "feature_contract.json"
write_json({
    "data_source": DATA_SOURCE,
    "source_stage": TRAINING_DATA_STAGE,
    "imputation_method": IMPUTATION_METHOD,
    "feature_count": len(feature_columns),
    "feature_columns": feature_columns,
    "feature_groups": feature_groups,
}, feature_contract_path)

metrics_path = OUTPUT_DIR / "metrics.json"
write_json({
    "model": "wildfire",
    "data_source": DATA_SOURCE,
    "source_stage": TRAINING_DATA_STAGE,
    "imputation_method": IMPUTATION_METHOD,
    "cell_subset": CELL_SUBSET,
    "cell_subset_info": {
        "categories": CELL_SUBSET_CATEGORIES[CELL_SUBSET],
        "n_cells_selected": len(SELECTED_TRAINING_CELLS),
        "n_archive_cells": SOURCE_CELLS,
        "fire_region_csv": CELL_SUBSET_INFO.get("fire_region_csv"),
    },
    "run_mode": MODE,
    "load_mode": LOAD_MODE,
    "device": DEVICE,
    "reported_gpu": GPU_NAME,
    "metric_definitions": {
        "pr_auc": "Average precision; ranking quality with rare positives",
        "roc_auc": "Separation of fire vs non-fire scores",
        "f1_at_0_5": "F1 when calibrated p_fire >= 0.5",
        "f1_at_25": "F1 treating daily top-25 alert_score cells as predicted positives",
        "precision_at_25": "Share of daily top-25 alerts that are true fires",
        "recall_at_25": "Share of all positives captured in daily top-25 alerts",
    },
    "data_quantity": {
        "loaded_rows": SOURCE_ROWS,
        "loaded_cells": SOURCE_CELLS,
        "loaded_days": SOURCE_DAYS,
        "loaded_positives": SOURCE_POSITIVES,
        "splits": split_quantity.to_dict(orient="records"),
    },
    "architecture": {
        "features": len(feature_columns),
        "classifier_pipeline": [IMPUTATION_METHOD, "LightGBM classifier"],
        "ranker_pipeline": [IMPUTATION_METHOD, "LightGBM ranker"],
        "daily_blend": {"classifier_percentile": 0.50, "ranker_percentile": 0.50},
    },
    "validation_2023": validation_metrics,
    "validation_2023_last14d": validation_last14_metrics,
    "note": "2025 / test.parquet is reserved for the inference notebook",
}, metrics_path)

loaded = joblib.load(model_path)
verification = validation.sort_values(["label_date", "cell_id"]).head(min(1000, len(validation)))
original_probability = classifier_pipeline.predict_proba(verification[feature_columns])[:, 1]
loaded_probability = loaded["classifier_pipeline"].predict_proba(verification[feature_columns])[:, 1]
original_rank = ranker_pipeline.predict(verification[feature_columns])
loaded_rank = loaded["ranker_pipeline"].predict(verification[feature_columns])
probability_difference = float(np.max(np.abs(original_probability - loaded_probability)))
rank_difference = float(np.max(np.abs(original_rank - loaded_rank)))
assert probability_difference <= 1e-12
assert rank_difference <= 1e-12

artifacts = [
    model_path, classifier_weights_path, ranker_weights_path, cell_subset_path,
    feature_contract_path, metrics_path,
    metrics_dir / "validation_2023.json",
    metrics_dir / "validation_2023_last14d.json",
    metrics_dir / "optuna_summary.json",
    quantity_plot, calibration_plot, risk_map, importance_plot,
]
artifacts = [path for path in artifacts if path.is_file()]
manifest_path = OUTPUT_DIR / "run_manifest.json"
write_json({
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model": "wildfire",
    "data_source": DATA_SOURCE,
    "source_stage": TRAINING_DATA_STAGE,
    "software": {
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__, "lightgbm": lgb.__version__,
    },
    "hardware": {"reported_gpu": GPU_NAME, "lightgbm_device": DEVICE},
    "inputs": {
        "data_root": str(DATA_ROOT),
        "load_mode": LOAD_MODE,
        "tables": [
            {"path": str(path), "sha256": sha256(path)} for path in TABLE_PARQUETS
        ],
        "fire_region_csv": (
            {"path": FIRE_REGION_CSV_PATH, "sha256": sha256(FIRE_REGION_CSV_PATH)}
            if FIRE_REGION_CSV_PATH is not None else None
        ),
        "meta_json": {"path": META_JSON, "sha256": sha256(META_JSON)},
        "dataset_metadata_json": {"path": DATASET_METADATA_JSON, "sha256": sha256(DATASET_METADATA_JSON)},
        "feature_columns_json": {"path": FEATURE_COLUMNS_JSON, "sha256": sha256(FEATURE_COLUMNS_JSON)},
    },
    "artifacts": {
        path.name: {"path": path, "size_bytes": path.stat().st_size, "sha256": sha256(path)}
        for path in artifacts
    },
    "reload_check": {
        "rows": len(verification),
        "maximum_classifier_difference": probability_difference,
        "maximum_ranker_difference": rank_difference,
    },
}, manifest_path)

print("Training outputs (attach this notebook output as Input to inference):")
for path in (model_path, classifier_weights_path, ranker_weights_path, feature_contract_path, metrics_path, manifest_path):
    print(f"  {path}")
print(f"Artifact reload passed on {len(verification):,} validation rows with zero difference.")
print("Note: inference notebook update later should load models/wildfire_model.joblib")


## Handoff

1. Confirm `models/wildfire_model.joblib` exists in the output folder.
2. **Save Version** on Kaggle.
3. Attach that notebook output to the inference notebook (update inference later to load `wildfire_model.joblib`).
4. Attach matching history pack (`california-wildfire-knn` or `...-median`). Use `test.parquet` / 2025 only in inference.
